In [11]:
import os, sys, warnings
from pathlib import Path
import torch

import tempfile
import numpy as np
import pandas as pd
import torch
from types import SimpleNamespace

NOTEBOOK_DIR = Path().resolve()
SRC_DIR      = NOTEBOOK_DIR.parent / "src" 

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences, heterogeneous_sampler
from dataloaders._ts_dataloader import DataLoaderFactory
from _test_utils import make_df, make_mcfg, make_entry, make_dcfg, setup_test_data, remove_test_data

warnings.filterwarnings("ignore")
print("imports OK")

imports OK


## Test — `fork_sequences` unit tests

In [ ]:
def make_batch(B=2, T=200, C=3, Vh=1, H=6, device="cpu"):
    """Minimal collated batch as _full_series_collate_fn would produce."""
    return {
        "x_enc":          torch.randn(B, T, C, 1 + Vh, device=device),
        "available_mask": torch.ones(B, T, C, device=device),   # [B, T, C]
        "channel_mask":   torch.ones(B, C, device=device),
        "horizon":        torch.full((B,), H, dtype=torch.long, device=device),
    }

L, H, S = 32, 6, 1

# ── Training path (fcd_samples=4) ─────────────────────────────────────────────
print("\n--- Training path (fcd_samples=4) ---")
batch = make_batch(B=2, T=200, C=3, Vh=1, H=H)
out = fork_sequences(batch, context_length=L, fcd_samples=4, horizon=H)
B_out, enc_size, C_out, feat = out["insample_y"].shape
print(f"insample_y:    {tuple(out['insample_y'].shape)}    expected [2, {enc_size}, 3, 2]")
print(f"outsample_y:   {tuple(out['outsample_y'].shape)}   expected [2, 4, {H}, 3]")
print(f"available_mask:{tuple(out['available_mask'].shape)}")
assert out["outsample_y"].shape == (2, 4, H, 3), \
    f"outsample_y shape wrong: {out['outsample_y'].shape}"
assert out["available_mask"].shape == (2, enc_size, 3), \
    f"available_mask shape wrong: {out['available_mask'].shape}"
print("training path shapes ✓")

# ── Val/test path (fcd_samples=-1) ────────────────────────────────────────────
print("\n--- Val/test path (fcd_samples=-1) ---")
T = 200
batch_val  = make_batch(B=2, T=T, C=3, Vh=0, H=H)
out_val    = fork_sequences(batch_val, context_length=L, fcd_samples=-1, horizon=H)
n_expected = T - L - H + 1
print(f"n_valid_fcds({T}, L={L}, H={H}, S={S}) = {n_expected}")
print(f"outsample_y:   {tuple(out_val['outsample_y'].shape)}  expected [2, {n_expected}, {H}, 3]")
assert out_val["outsample_y"].shape[1] == n_expected, \
    f"n_fcds wrong: {out_val['outsample_y'].shape[1]} != {n_expected}"
print("val/test path shapes ✓")

# ── Left-padding: windows never start in padding ───────────────────────────────
print("\n--- Left-padding: sampler skips padded timesteps ---")
T_real, T_pad = 100, 50
T_total       = T_real + T_pad
mask = torch.zeros(1, T_total, 3)       # [B, T, C]
mask[:, T_pad:, :] = 1.0               # real data starts at T_pad
batch_pad = {
    "x_enc":          torch.randn(1, T_total, 3, 1),
    "available_mask": mask,
    "horizon":        torch.tensor([H]),
}

for _ in range(20):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=4, horizon=H)
    assert ws[0].item() >= T_pad, \
        f"window_start {ws[0].item()} < T_pad {T_pad} — sampled in padding!"
print("all 20 sampled window_starts ≥ T_pad ✓")

# ── available_mask shape assertion ────────────────────────────────────────────
print("\n--- available_mask shape mismatch raises AssertionError ---")
bad_mask_batch = {
    "x_enc":          torch.randn(2, 100, 3, 1),
    "available_mask": torch.ones(2, 3, 100),   # wrong: [B, C, T] instead of [B, T, C]
    "horizon":        torch.tensor([H, H]),
}
try:
    fork_sequences(bad_mask_batch, context_length=L, fcd_samples=4, horizon=H)
    assert False, "should have raised"
except AssertionError as e:
    print(f"raised AssertionError as expected: {e}")

print("\n✓ TEST PASSED")


--- Training path (fcd_samples=4) ---
insample_y:    (2, 35, 3, 2)    expected [2, 35, 3, 2]
outsample_y:   (2, 4, 6, 3)   expected [2, 4, 6, 3]
available_mask:(2, 35, 3)
training path shapes ✓

--- Val/test path (fcd_samples=-1) ---
n_valid_fcds(200, L=32, H=6, S=1) = 163
outsample_y:   (2, 163, 6, 3)  expected [2, 163, 6, 3]
val/test path shapes ✓

--- Left-padding: sampler skips padded timesteps ---
all 20 sampled window_starts ≥ T_pad ✓

--- available_mask shape mismatch raises AssertionError ---
raised AssertionError as expected: available_mask must be [B, S, C] = [2, 100, 3], got (2, 3, 100)

✓ TEST PASSED


## Test: window_sampling start timestamp must never exceeds new max_start

In [13]:
L, H, T = 8, 6, 30
mask    = torch.ones(2, T, 3)
_, block_len = heterogeneous_sampler(mask, L, fcd_samples=1, horizon=H)
new_max_start = T - block_len

overflow_found = False
for _ in range(100):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=1, horizon=H)
    if not (ws <= new_max_start).all().item():
        overflow_found = True
        print(f"  overflow: ws={ws.tolist()}, max={new_max_start}")
        break

if not overflow_found:
    print("\n✓ TEST PASSED")


✓ TEST PASSED


## TEST - Check masks

In [6]:
def print_masks(outsample_mask, insample_mask, L, H, t_offset=0, max_windows=None):
    """
    Print insample and outsample masks side by side for series 0, channel 0.
    Each row: [context mask (L) | horizon mask (H)]
    """
    n_fcds   = outsample_mask.shape[1]
    enc_size = insample_mask.shape[1]
    n        = min(n_fcds, max_windows) if max_windows else n_fcds
    print(f"\n  (showing {n}/{n_fcds} windows,  L={L}  H={H})")
    print(f"  {'w':>4}  t_range         insample(L={L})  outsample(H={H})")
    print(f"  {''  :>4}  {''  :14}  {'-'*L}  {'-'*H}")
    for w in range(n):
        t_start  = w + t_offset
        t_end    = w + L + H - 1 + t_offset
        in_vals  = insample_mask[0, w : w + L, 0].tolist() if w + L <= enc_size else []
        in_str   = "".join(str(int(v)) for v in in_vals).ljust(L)
        out_vals = outsample_mask[0, w, :, 0].tolist()
        out_str  = "".join(str(int(v)) for v in out_vals)
        print(f"  {w:>4}  t=[{t_start:3d},{t_end:3d}]  {in_str}  {out_str}")

print("helpers OK")

helpers OK


In [16]:
TEST_DATA_DIR = os.path.join(os.getcwd(), "test_data")
setup_test_data(TEST_DATA_DIR)

L, H      = 6, 3
val_size  = 9
test_size = 9

try:
    csv = f"{TEST_DATA_DIR}/test.csv"
    make_df(n_series=1, n_steps=30).to_csv(csv, index=False)

    mcfg = make_mcfg(
        model_name='linear',
        context_length=L, 
        h=H
    )
    entry   = make_entry(csv, name="ds", horizon=H,
                         val_size=val_size, test_size=test_size,
                         use_context_head=True)
    dcfg    = make_dcfg([entry])
    factory = DataLoaderFactory(mcfg, dcfg)

    train_loader = factory.train_dataloader()
    val_loader   = factory.val_dataloaders()["val"]
    test_loader  = factory.test_dataloaders()["test"]

    n_steps  = 30   # must match make_df(n_steps=...)
    T_train  = n_steps - val_size - test_size   # 12
    T_val    = val_size                          # 9
    T_test   = test_size                         # 9

    # absolute start of each dataset in the full series
    # val/test prepend L ctx rows, so their series starts L before their split
    t_offset_train = 0
    t_offset_val   = T_train - L     # ctx starts at last L rows of train
    t_offset_test  = T_train + T_val - L

    # ── Train ─────────────────────────────────────────────────────────────────
    # Layout: [train=1 | H-1 ext=0]
    # All windows fully inside train → mask=1
    # Last H-1 windows overlap ext → partly masked

    print("\n" + "="*60)
    print(f"TRAIN  layout: [train=1 | ext=0]  (ext={H-1} rows)")
    print("="*60)
    for batch in train_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]   # [B, n_fcds, H, C]
        avail          = batch["available_mask"]  # [B, S, C]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_train)

        # outsample_mask[b,w,h,c] must equal avail[b, w+L+h, c]
        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        if len(bad) == 0:
            print("\n✓ TEST PASSED")

        # last H-1 windows should be partially masked (overlap ext)
        n_fcds = outsample_mask.shape[1]
        boundary_partly_masked = any(
            (outsample_mask[0, w, :, 0] == 0).any().item()
            for w in range(max(0, n_fcds - (H - 1)), n_fcds)
        )
        if boundary_partly_masked:
            print("\n✓ TEST PASSED")
        break

    # ── Val ───────────────────────────────────────────────────────────────────
    # Layout: [ctx=0 | val=1 | H-1 ext=0]
    # Windows whose horizons are fully inside val → all mask=1
    # Windows overlapping ctx or ext → partly/fully masked

    print("\n" + "="*60)
    print(f"VAL    layout: [ctx=0 | val=1 | ext=0]  (ctx={L}, ext={H-1} rows)")
    print("="*60)
    for batch in val_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]
        avail          = batch["available_mask"]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_val)

        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        if len(bad) == 0:
            print("\n✓ TEST PASSED")

        n_fcds = outsample_mask.shape[1]
        # windows fully inside val: h_start>=T_ctx and h_end<T_ctx+val_size
        # T_ctx=L (ctx rows), so h_start = w+L >= L → w >= 0 always
        # h_end = w+L+H-1 < L+val_size → w < val_size-H+1
        fully_unmasked = [w for w in range(n_fcds)
                          if (outsample_mask[0, w, :, 0] == 1).all().item()]
        expected_fully_unmasked = val_size - H + 1
        if len(fully_unmasked) == expected_fully_unmasked:
            print("\n✓ TEST PASSED")

        # last H-1 windows should be partially masked (overlap ext)
        boundary_partly_masked = any(
            (outsample_mask[0, w, :, 0] == 0).any().item()
            for w in range(max(0, n_fcds - (H - 1)), n_fcds)
        )
        if boundary_partly_masked:
            print("\n✓ TEST PASSED")
        break

    # ── Test ──────────────────────────────────────────────────────────────────
    # Layout: [ctx=0 | test=1]
    # No extension — windows fully inside test → all mask=1
    # Early windows overlapping ctx → partly masked

    print("\n" + "="*60)
    print(f"TEST   layout: [ctx=0 | test=1]  (ctx={L}, no ext)")
    print("="*60)
    for batch in test_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]
        avail          = batch["available_mask"]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_test)

        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        if len(bad) == 0:
            print("\n✓ TEST PASSED")

        n_fcds = outsample_mask.shape[1]
        fully_unmasked = [w for w in range(n_fcds)
                          if (outsample_mask[0, w, :, 0] == 1).all().item()]
        expected_fully_unmasked = test_size - H + 1
        if len(fully_unmasked) == expected_fully_unmasked:
            print("\n✓ TEST PASSED")

        # no extension rows → last window should be fully unmasked
        if (outsample_mask[0, -1, :, 0] == 1).all().item():
            print("\n✓ TEST PASSED")
        break
finally:
    remove_test_data(TEST_DATA_DIR)


TRAIN  layout: [train=1 | ext=0]  (ext=2 rows)

  (showing 6/6 windows,  L=6  H=3)
     w  t_range         insample(L=6)  outsample(H=3)
                        ------  ---
     0  t=[  0,  8]  111111  111
     1  t=[  1,  9]  111111  111
     2  t=[  2, 10]  111111  111
     3  t=[  3, 11]  111111  111
     4  t=[  4, 12]  111111  110
     5  t=[  5, 13]  111111  100

✓ TEST PASSED

✓ TEST PASSED

VAL    layout: [ctx=0 | val=1 | ext=0]  (ctx=6, ext=2 rows)

  (showing 9/9 windows,  L=6  H=3)
     w  t_range         insample(L=6)  outsample(H=3)
                        ------  ---
     0  t=[  6, 14]  000000  111
     1  t=[  7, 15]  000001  111
     2  t=[  8, 16]  000011  111
     3  t=[  9, 17]  000111  111
     4  t=[ 10, 18]  001111  111
     5  t=[ 11, 19]  011111  111
     6  t=[ 12, 20]  111111  111
     7  t=[ 13, 21]  111111  110
     8  t=[ 14, 22]  111111  100

✓ TEST PASSED

✓ TEST PASSED

✓ TEST PASSED

TEST   layout: [ctx=0 | test=1]  (ctx=6, no ext)

  (showing 7/7 win